In [30]:
import sys
sys.path.append('../')
import datasets
import log_reg
from dataproc import extract_wvs
# from dataproc import get_discharge_summaries
from dataproc import concat_and_split
from dataproc import build_vocab
from dataproc import vocab_index_descriptions
from dataproc import word_embeddings
from constants import MIMIC_3_DIR, DATA_DIR
from nltk.tokenize import RegexpTokenizer, sent_tokenize
from tqdm import tqdm

import numpy as np
import pandas as pd

from collections import Counter, defaultdict
import csv
import math
import operator

Let's do some data processing in a much better way, with a notebook.

First, let's define some stuff.

In [38]:
Y = 'full' #use all available labels in the dataset for prediction
notes_file = '%s/NOTEEVENTS.csv' % MIMIC_3_DIR # raw note events downloaded from MIMIC-III
vocab_size = 'full' #don't limit the vocab size to a specific number
vocab_min = 3 #discard tokens appearing in fewer than this many documents

# df = pd.read_csv(notes_file)
# with pd.option_context('display.max_colwidth', None):  # Temporarily set max column width to None
#     print(df['TEXT'].head(1))

/var/folders/ht/01r_hjk551jbd48xj1kkm7kh0000gp/T/ipykernel_27577/202364405.py:6: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(notes_file)


0    Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]\n\n\nService:\nADDENDUM:\n\nRADIOLOGIC STUDIES:  Radiologic studies also included a chest\nCT, which confirmed cavitary lesions in the left lung apex\nconsistent with infectious process/tuberculosis.  This also\nmoderate-sized left pleural effusion.\n\nHEAD CT:  Head CT showed no intracranial hemorrhage or mass\neffect, but old infarction consistent with past medical\nhistory.\n\nABDOMINAL CT:  Abdominal CT showed lesions of\nT10 and sacrum most likely secondary to osteoporosis. These can\nbe followed by repeat imaging as an outpatient.\n\n\n\n                            [**First Name8 (NamePattern2) **] [**First Name4 (NamePattern1) 1775**] [**Last Name (NamePattern1) **], M.D.  [**MD Number(1) 1776**]\n\nDictated By:[**Hospital 1807**]\nMEDQUIST36\n\nD:  [**2151-8-5**]  12:11\nT:  [**2151-8-5**]  12:21\nJOB#:  [**Job Number 1808**]\n
Name: TEXT, dtype: object


# Data processing

## Combine diagnosis and procedure codes and reformat them

The codes in MIMIC-III are given in separate files for procedures and diagnoses, and the codes are given without periods, which might lead to collisions if we naively combine them. So we have to add the periods back in the right place.

In [63]:
dfproc = pd.read_csv('%s/PROCEDURES_ICD.csv' % MIMIC_3_DIR)
dfdiag = pd.read_csv('%s/DIAGNOSES_ICD.csv' % MIMIC_3_DIR)

In [64]:
dfdiag['absolute_code'] = dfdiag.apply(lambda row: str(datasets.reformat(str(row[4]), True)), axis=1)
dfproc['absolute_code'] = dfproc.apply(lambda row: str(datasets.reformat(str(row[4]), False)), axis=1)

/var/folders/ht/01r_hjk551jbd48xj1kkm7kh0000gp/T/ipykernel_27577/2369496630.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfdiag['absolute_code'] = dfdiag.apply(lambda row: str(datasets.reformat(str(row[4]), True)), axis=1)
/var/folders/ht/01r_hjk551jbd48xj1kkm7kh0000gp/T/ipykernel_27577/2369496630.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfproc['absolute_code'] = dfproc.apply(lambda row: str(datasets.reformat(str(row[4]), False)), axis=1)


In [65]:
dfcodes = pd.concat([dfdiag, dfproc])

In [66]:
dfcodes.to_csv('%s/ALL_CODES.csv' % MIMIC_3_DIR, index=False,
               columns=['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'SEQ_NUM', 'absolute_code'],
               header=['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'SEQ_NUM', 'ICD9_CODE'])

## How many codes are there?

In [67]:
#In the full dataset (not just discharge summaries)
df = pd.read_csv('%s/ALL_CODES.csv' % MIMIC_3_DIR, dtype={"ICD9_CODE": str})
len(df['ICD9_CODE'].unique())

8994

## Tokenize and preprocess raw text

Preprocessing time!

This will:
- Select only discharge summaries and their addenda
- remove punctuation and numeric-only tokens, removing 500 but keeping 250mg
- lowercase all tokens

In [23]:
import nltk
# nltk.download('punkt_tab')

# from nltk.tokenize.punkt import PunktSentenceTokenizer
# tokenizer = PunktSentenceTokenizer()
# sentences = tokenizer.tokenize("Your text here.")
# print(sentences)

from nltk.tokenize import sent_tokenize

#try sentence split
text = "Dr. Smith is a renowned cardiologist. He works at St. Mary's Hospital."
sentences = sent_tokenize(text)
print(sentences)

['Dr. Smith is a renowned cardiologist.', "He works at St. Mary's Hospital."]


In [68]:
#retain only alphanumeric
tokenizer = RegexpTokenizer(r'\w+')

def write_discharge_summaries(out_file):
    notes_file = '%s/NOTEEVENTS.csv' % (MIMIC_3_DIR)
    print("processing notes file")
    with open(notes_file, 'r') as csvfile:
        with open(out_file, 'w') as outfile:
            print("writing to %s" % (out_file))
            outfile.write(','.join(['SUBJECT_ID', 'HADM_ID', 'CHARTTIME', 'TEXT']) + '\n')
            notereader = csv.reader(csvfile)
            #header
            next(notereader)
            i = 0
            for line in tqdm(notereader):
                subj = int(line[1])
                category = line[6]
                if category == "Discharge summary":
                    note = line[10]
                    # Tokenize and clean each sentence
                    sentences = sent_tokenize(note)  # Split into sentences
                    cleaned_sentences = []
                    for sentence in sentences:
                        # Tokenize, lowercase, and remove numerics in each sentence
                        tokens = [t.lower() for t in tokenizer.tokenize(sentence) if not t.isnumeric()]
                        cleaned_sentence = ' '.join(tokens)
                        cleaned_sentences.append(cleaned_sentence)
                    # Join sentences with '\n\n' to mark end of each sentence
                    text = '"' + '||'.join(cleaned_sentences) + '"'
                    outfile.write(','.join([line[1], line[2], line[4], text]) + '\n')
                i += 1
    return out_file

In [69]:
#This reads all notes, selects only the discharge summaries, and tokenizes them, returning the output filename
# disch_full_file = get_discharge_summaries.write_discharge_summaries(out_file="%s/disch_full.csv" % MIMIC_3_DIR)
disch_full_file = write_discharge_summaries(out_file="%s/disch_full_sent_split.csv" % MIMIC_3_DIR)

processing notes file
writing to /Users/zhirnikovich/Desktop/Study/CS769_Advanced_NLP/Project/caml-mimic-master/mimicdata/mimic3/disch_full_sent_split.csv


0it [00:00, ?it/s]

2083180it [04:45, 7285.76it/s] 


Let's read this in and see what kind of data we're working with

In [70]:
# df = pd.read_csv('%s/disch_full.csv' % MIMIC_3_DIR)
df = pd.read_csv('%s/disch_full_sent_split.csv' % MIMIC_3_DIR)

In [71]:
#How many admissions?
len(df['HADM_ID'].unique())
#Check split for the first note
with pd.option_context('display.max_colwidth', None):  # Temporarily set max column width to None
    print(df['TEXT'].head(1))

0    admission date discharge date service addendum radiologic studies radiologic studies also included a chest ct which confirmed cavitary lesions in the left lung apex consistent with infectious process tuberculosis||this also moderate sized left pleural effusion||head ct head ct showed no intracranial hemorrhage or mass effect but old infarction consistent with past medical history||abdominal ct abdominal ct showed lesions of t10 and sacrum most likely secondary to osteoporosis||these can be followed by repeat imaging as an outpatient||first name8 namepattern2 first name4 namepattern1 last name namepattern1 m d||md number dictated by hospital medquist36 d t job job number
Name: TEXT, dtype: object


In [72]:
#Tokens and types
types = set()
num_tok = 0
for row in df.itertuples():
    for w in row[4].split():
        types.add(w)
        num_tok += 1

In [73]:
print("Num types", len(types))
print("Num tokens", str(num_tok))

Num types 1383022
Num tokens 74088388


In [74]:
#Let's sort by SUBJECT_ID and HADM_ID to make a correspondence with the MIMIC-3 label file
df = df.sort_values(['SUBJECT_ID', 'HADM_ID'])

In [75]:
#Sort the label file by the same
dfl = pd.read_csv('%s/ALL_CODES.csv' % MIMIC_3_DIR)
dfl = dfl.sort_values(['SUBJECT_ID', 'HADM_ID'])

/var/folders/ht/01r_hjk551jbd48xj1kkm7kh0000gp/T/ipykernel_27577/1618252027.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  dfl = pd.read_csv('%s/ALL_CODES.csv' % MIMIC_3_DIR)


In [76]:
len(df['HADM_ID'].unique()), len(dfl['HADM_ID'].unique())

(52726, 58976)

## Consolidate labels with set of discharge summaries

Looks like there were some HADM_ID's that didn't have discharge summaries, so they weren't included with our notes

In [77]:
#Let's filter out these HADM_ID's
hadm_ids = set(df['HADM_ID'])
with open('%s/ALL_CODES.csv' % MIMIC_3_DIR, 'r') as lf:
    with open('%s/ALL_CODES_filtered.csv' % MIMIC_3_DIR, 'w') as of:
        w = csv.writer(of)
        w.writerow(['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE', 'ADMITTIME', 'DISCHTIME'])
        r = csv.reader(lf)
        #header
        next(r)
        for i,row in enumerate(r):
            hadm_id = int(row[2])
            #print(hadm_id)
            #break
            if hadm_id in hadm_ids:
                w.writerow(row[1:3] + [row[-1], '', ''])

In [78]:
dfl = pd.read_csv('%s/ALL_CODES_filtered.csv' % MIMIC_3_DIR, index_col=None)

/var/folders/ht/01r_hjk551jbd48xj1kkm7kh0000gp/T/ipykernel_27577/1742194954.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfl = pd.read_csv('%s/ALL_CODES_filtered.csv' % MIMIC_3_DIR, index_col=None)


In [79]:
len(dfl['HADM_ID'].unique())

52726

In [80]:
#we still need to sort it by HADM_ID
dfl = dfl.sort_values(['SUBJECT_ID', 'HADM_ID'])
dfl.to_csv('%s/ALL_CODES_filtered.csv' % MIMIC_3_DIR, index=False)

## Append labels to notes in a single file

In [81]:
#Now let's append each instance with all of its codes
#this is pretty non-trivial so let's use this script I wrote, which requires the notes to be written to file
# sorted_file = '%s/disch_full.csv' % MIMIC_3_DIR
sorted_file = '%s/disch_full_sent_split.csv' % MIMIC_3_DIR
df.to_csv(sorted_file, index=False)

In [82]:
dlllf = pd.read_csv('%s/disch_full_sent_split.csv' % MIMIC_3_DIR)
#Check sent split for the first sorted note
with pd.option_context('display.max_colwidth', None):  # Temporarily set max column width to None
    print(dlllf['TEXT'].head(1))

0    admission date discharge date date of birth sex m service medicine chief complaint admitted from rehabilitation for hypotension systolic blood pressure to the 70s and decreased urine output||history of present illness the patient is a year old male who had been hospitalized at the hospital1 from through of after undergoing a left femoral at bypass graft and was subsequently discharged to a rehabilitation facility||on he presented again to the hospital1 after being found to have a systolic blood pressure in the 70s and no urine output for hours||a foley catheter placed at the rehabilitation facility yielded cc of murky brown urine||there may also have been purulent discharge at the penile meatus at this time||on presentation to the emergency department the patient was without subjective complaints||in the emergency department he was found to have systolic blood pressure of||he was given liters of intravenous fluids and transiently started on dopamine for a systolic blood pressure i

In [ ]:
# dooof = pd.read_csv('%s/disch_full.csv' % MIMIC_3_DIR)
# with pd.option_context('display.max_colwidth', None):  # Temporarily set max column width to None
#     print(dooof['TEXT'].head(1))

In [83]:
labeled = concat_and_split.concat_data('%s/ALL_CODES_filtered.csv' % MIMIC_3_DIR, sorted_file)

CONCATENATING
0 done
10000 done
20000 done
30000 done
40000 done
50000 done


In [84]:
#name of the file we just made
print(labeled)

/Users/zhirnikovich/Desktop/Study/CS769_Advanced_NLP/Project/caml-mimic-master/mimicdata/mimic3/notes_labeled.csv


Let's sanity check the combined data we just made. Do we have all hadm id's accounted for, and the same vocab stats?

In [85]:
dfnl = pd.read_csv(labeled)
#Tokens and types
types = set()
num_tok = 0
for row in dfnl.itertuples():
    for w in row[3].split():
        types.add(w)
        num_tok += 1

In [86]:
print("num types", len(types), "num tokens", num_tok)

num types 1383022 num tokens 74088388


In [87]:
len(dfnl['HADM_ID'].unique())

52726

## Create train/dev/test splits

In [88]:
fname = '%s/notes_labeled.csv' % MIMIC_3_DIR
base_name = "%s/disch_sent_split" % MIMIC_3_DIR #for output
tr, dv, te = concat_and_split.split_data(fname, base_name=base_name)

SPLITTING
0 read
10000 read
20000 read
30000 read
40000 read
50000 read


## Build vocabulary from training data

In [ ]:
vocab_min = 3
vname = '%s/vocab.csv' % MIMIC_3_DIR
build_vocab.build_vocab(vocab_min, tr, vname)

## Sort each data split by length for batching

In [91]:
for splt in ['train', 'dev', 'test']:
    filename = '%s/disch_sent_split_%s_split.csv' % (MIMIC_3_DIR, splt)
    df = pd.read_csv(filename)
    df['length'] = df.apply(lambda row: len(str(row['TEXT']).split()), axis=1)
    df = df.sort_values(['length'])
    df.to_csv('%s/%s_sent_split_full.csv' % (MIMIC_3_DIR, splt), index=False)

## Pre-train word embeddings

Let's train word embeddings on all words

In [29]:
w2v_file = word_embeddings.word_embeddings('full', '%s/disch_full.csv' % MIMIC_3_DIR, 100, 0, 5)

building word2vec vocab on /nethome/jmullenbach3/replication/cnn-medical-text/mimicdata/mimic3//disch_full.csv...
training...
writing embeddings to /nethome/jmullenbach3/replication/cnn-medical-text/mimicdata/mimic3//processed_full.w2v


## Write pre-trained word embeddings with new vocab

In [30]:
extract_wvs.gensim_to_embeddings('%s/processed_full.w2v' % MIMIC_3_DIR, '%s/vocab.csv' % MIMIC_3_DIR, Y)

100%|██████████| 51917/51917 [02:58<00:00, 290.28it/s]


## Pre-process code descriptions using the vocab

In [31]:
vocab_index_descriptions.vocab_index_descriptions('%s/vocab.csv' % MIMIC_3_DIR,
                                                  '%s/description_vectors.vocab' % MIMIC_3_DIR)

100%|██████████| 22267/22267 [00:00<00:00, 62940.71it/s]


## Filter each split to the top 50 diagnosis/procedure codes

In [92]:
Y = 50

In [93]:
#first calculate the top k
counts = Counter()
dfnl = pd.read_csv('%s/notes_labeled.csv' % MIMIC_3_DIR)
for row in dfnl.itertuples():
    for label in str(row[4]).split(';'):
        counts[label] += 1

In [94]:
codes_50 = sorted(counts.items(), key=operator.itemgetter(1), reverse=True)

In [95]:
codes_50 = [code[0] for code in codes_50[:Y]]

In [96]:
codes_50

['401.9',
 '38.93',
 '428.0',
 '427.31',
 '414.01',
 '96.04',
 '96.6',
 '584.9',
 '250.00',
 '96.71',
 '272.4',
 '518.81',
 '99.04',
 '39.61',
 '599.0',
 '530.81',
 '96.72',
 '272.0',
 '285.9',
 '88.56',
 '244.9',
 '486',
 '38.91',
 '285.1',
 '36.15',
 '276.2',
 '496',
 '99.15',
 '995.92',
 'V58.61',
 '507.0',
 '038.9',
 '88.72',
 '585.9',
 '403.90',
 '311',
 '305.1',
 '37.22',
 '412',
 '33.24',
 '39.95',
 '287.5',
 '410.71',
 '276.1',
 'V45.81',
 '424.0',
 '45.13',
 'V15.82',
 '511.9',
 '37.23']

In [97]:
with open('%s/TOP_%s_CODES.csv' % (MIMIC_3_DIR, str(Y)), 'w') as of:
    w = csv.writer(of)
    for code in codes_50:
        w.writerow([code])

In [98]:
for splt in ['train', 'dev', 'test']:
    print(splt)
    hadm_ids = set()
    with open('%s/%s_50_hadm_ids.csv' % (MIMIC_3_DIR, splt), 'r') as f:
        for line in f:
            hadm_ids.add(line.rstrip())
    with open('%s/notes_labeled.csv' % MIMIC_3_DIR, 'r') as f:
        with open('%s/%s_%s.csv' % (MIMIC_3_DIR, splt, str(Y)), 'w') as of:
            r = csv.reader(f)
            w = csv.writer(of)
            #header
            w.writerow(next(r))
            i = 0
            for row in r:
                hadm_id = row[1]
                if hadm_id not in hadm_ids:
                    continue
                codes = set(str(row[3]).split(';'))
                filtered_codes = codes.intersection(set(codes_50))
                if len(filtered_codes) > 0:
                    w.writerow(row[:3] + [';'.join(filtered_codes)])
                    i += 1

train
dev
test


In [99]:
for splt in ['train', 'dev', 'test']:
    filename = '%s/%s_%s.csv' % (MIMIC_3_DIR, splt, str(Y))
    df = pd.read_csv(filename)
    df['length'] = df.apply(lambda row: len(str(row['TEXT']).split()), axis=1)
    df = df.sort_values(['length'])
    df.to_csv('%s/%s_%s_sent_split.csv' % (MIMIC_3_DIR, splt, str(Y)), index=False)

In [44]:
for splt in ['train', 'dev', 'test']:
    filename = '%s/%s_%s.csv' % (MIMIC_3_DIR, splt, str(Y))
    df = pd.read_csv(filename)
    # print(df.head())
    # print(df.columns)
    df['LABELS'] = df['LABELS'].str.replace(';', ' ')
    df['united'] = df['TEXT'] + '__label__' + df['LABELS']
    # df['united'] = df.iloc[:, 2] + '__label__' + df.iloc[:, 3]
    df['united'].to_csv('%s/%s_%s_th0.txt' % (MIMIC_3_DIR, splt, str(Y)), index=False, header=False)
